# 🤖 Comprendre le LLM HuggingFace et son mécanisme de cache

**Objectif** : répondre en profondeur à 4 questions —
1. Qu'est-ce qu'un LLM, concrètement, à l'intérieur ?
2. Comment un LLM hébergé sur HuggingFace fonctionne-t-il (routage, providers) ?
3. Qu'est-ce que "le cache" dont on parle dans ce projet — et est-ce vraiment un mécanisme du LLM lui-même ?
4. Comment forcer le système à produire une explication réellement pertinente sur la prédiction du classifieur/régresseur, plutôt qu'un texte générique ou périmé ?


---
## 1. Qu'est-ce qu'un LLM — la mécanique interne, en profondeur

Un LLM (Large Language Model) comme `meta-llama/Llama-3.3-70B-Instruct` n'est **pas une base de données de phrases toutes faites**. C'est un réseau de neurones entraîné à faire une seule chose très bien : **prédire le mot suivant**, étant donné tout ce qui précède.

### Le pipeline interne, étape par étape

```
Texte d'entrée (ton prompt)
        ↓
1. TOKENIZATION
   Le texte est découpé en "tokens" — pas forcément des mots entiers.
   "Quantité suggérée" -> ["Quant", "ité", " sugg", "érée"]  (exemple simplifié)
        ↓
2. EMBEDDING
   Chaque token est converti en un vecteur de nombres (ex: 4096 dimensions)
   qui capture son "sens" dans un espace mathématique.
        ↓
3. ATTENTION (le cœur du Transformer)
   Pour chaque token, le modèle calcule à quel point il doit "prêter attention"
   à chacun des tokens précédents pour comprendre le contexte.
   C'est ce qui permet au modèle de savoir que "il" dans une phrase fait
   référence à "le client" mentionné 10 mots plus tôt.
        ↓
4. PLUSIEURS COUCHES (70 milliards de paramètres pour Llama-3.3-70B)
   L'information traverse des dizaines de couches d'attention + réseaux
   de neurones, raffinant progressivement la représentation du contexte.
        ↓
5. PRÉDICTION DU TOKEN SUIVANT
   La dernière couche produit une probabilité pour CHAQUE token possible
   du vocabulaire (souvent ~128 000 tokens différents) :
   "Le client va probablement" -> {"racheter": 0.34, "commander": 0.21, "aimer": 0.02, ...}
        ↓
6. ÉCHANTILLONNAGE (sampling)
   Un token est choisi selon ces probabilités, influencé par `temperature`
   et `top_p` (les paramètres qu'on envoie dans le payload de l'API).
        ↓
7. RÉPÉTITION — le token choisi est rajouté au texte, et on recommence
   l'étape 3 à 6 pour générer le token SUIVANT, jusqu'à un token de fin
   ou la limite `max_tokens`.
```

**Point essentiel** : le LLM ne "sait" jamais à l'avance toute la phrase qu'il va écrire. Il génère un token à la fois, en boucle, chaque nouveau token étant conditionné par tout ce qui a été généré avant. C'est pour ça qu'on parle de modèle **autorégressif**.


### Ce que `temperature` et `top_p` contrôlent réellement

Dans ton `explanation.py` :
```python
"temperature": 0.3,
"top_p": 0.9,
```

- **`temperature`** — aplatit ou accentue la distribution de probabilités avant l'échantillonnage. `temperature=0.3` (bas) rend le modèle très "sûr de lui" — il choisit presque toujours le token le plus probable, donc des réponses plus déterministes et répétitives. `temperature=1.0` donnerait des réponses plus variées et créatives, mais aussi plus risquées niveau cohérence.
- **`top_p`** — ne considère que les tokens dont la probabilité cumulée atteint 90% (`top_p=0.9`), en ignorant la longue traîne de tokens très improbables. Ça évite que le modèle "dérape" vers un mot totalement hors sujet.

**Implication concrète pour ton projet** : avec `temperature=0.3`, tes explications LLM seront volontairement assez uniformes d'un appel à l'autre pour un contexte similaire — ce n'est pas un défaut, c'est un choix cohérent avec un besoin business de textes professionnels et prévisibles plutôt que créatifs.


---
## 2. Un LLM "hébergé sur HuggingFace" — comment ça marche réellement

C'est le point le plus mal compris dans la session précédente, donc à bien clarifier.

**HuggingFace lui-même n'exécute quasiment jamais les modèles.** HuggingFace est devenu un **routeur** — comme tu l'as découvert en debuggant l'ancien endpoint mort. Quand ton code appelle :

```python
api_url = "https://router.huggingface.co/v1/chat/completions"
payload = {"model": "meta-llama/Llama-3.3-70B-Instruct:fastest", ...}
```

Voici ce qui se passe réellement :

```
Ton code Python
      ↓ HTTPS POST
router.huggingface.co  (juste un aiguilleur / load balancer intelligent)
      ↓ regarde le suffixe ":fastest"
      ↓ choisit le meilleur provider disponible pour CE modèle précis
      ↓ redirige la requête
Groq / Together / Fireworks / Cerebras / Novita / ...
      ↓ CE provider possède les vrais serveurs GPU (H100, etc.)
      ↓ exécute réellement les calculs du réseau de neurones décrits en Partie 1
      ↓ renvoie le texte généré
router.huggingface.co
      ↓ relaie la réponse
Ton code Python
```

Le log que tu as obtenu le prouve directement :
```json
"model": "llama-3.3-70b-versatile",
"x_groq": {"id": "req_01kyjfy3xqff7teybt72s08dh5", ...}
```

Le champ `x_groq` confirme que ta requête a été exécutée par **Groq**, pas par HuggingFace lui-même. HuggingFace a juste facturé/routé la requête vers le meilleur provider disponible à ce moment (`:fastest` = sélection automatique du provider le plus rapide).

**C'est pour ça que `Mistral-7B-Instruct-v0.3` a échoué** — aucun provider actif ne le servait plus (`'novita': {'status': 'error'}`), alors que `Llama-3.3-70B-Instruct` avait Groq disponible et fonctionnel.


---
## 3. Le "cache" — démystification complète

C'est ici que se trouve la vraie réponse à ta question *"pourquoi ça switch sur le cache"*. Il faut distinguer **3 caches complètement différents** qui n'ont presque rien à voir entre eux.


### Cache n°1 — Le KV-Cache interne du Transformer (à l'intérieur du modèle, invisible pour toi)

Pendant la génération autorégressive (Partie 1, étapes 3 à 7), recalculer l'attention sur TOUS les tokens précédents à CHAQUE nouveau token serait extrêmement coûteux en calcul. Les moteurs d'inférence (ceux que Groq/Together font tourner) utilisent un **KV-Cache** (Key-Value Cache) : ils stockent les représentations internes ("clés" et "valeurs" de l'attention) déjà calculées pour les tokens précédents, et ne recalculent que ce qui concerne le nouveau token.

```
Sans KV-cache : générer le token n°50 recalcule TOUT depuis le token n°1 -> très lent
Avec KV-cache : générer le token n°50 réutilise les calculs des tokens 1 à 49 -> rapide
```

**Ce cache est géré entièrement par l'infrastructure du provider (Groq, etc.).** Tu ne le contrôles pas, tu ne le vois jamais, et il n'a **aucun rapport** avec le fait que ton système affiche parfois un texte "répété". Il concerne uniquement la vitesse de génération DANS un seul appel API, pas entre deux appels différents.


### Cache n°2 — Le cache applicatif DE TON PROJET (`_explanation_cache`)

C'est **le seul cache pertinent** pour ce que tu observes, et il n'a rien à voir avec HuggingFace — c'est une décision de conception dans TON code, dans `explanation.py` :

```python
_explanation_cache: dict[str, tuple[str, datetime]] = {}

def _get_cache_key(client_id, code_article, month):
    raw_str = f"{client_id}_{code_article}_{month}"
    return hashlib.md5(raw_str.encode("utf-8")).hexdigest()
```

**Pourquoi ce cache existe** : appeler un LLM a un coût (latence de 1-3 secondes, et potentiellement facturation selon le provider). Si le même client revient sur le même produit dans le même mois, pas besoin de repayer/réattendre un nouvel appel — on réutilise le texte déjà généré. C'est une optimisation de performance légitime et standard.

**Mais voici le vrai problème, qu'on va reproduire concrètement ci-dessous.**


In [1]:
import hashlib

def get_cache_key(client_id, code_article, month):
    raw_str = f"{client_id}_{code_article}_{month}"
    return hashlib.md5(raw_str.encode("utf-8")).hexdigest()

# Scénario réel : le même client + même produit + même mois,
# mais le régresseur a produit deux quantités DIFFERENTES à deux moments différents
# (ex: pipeline relancé après un nouvel entraînement, ou deux requêtes API le même jour)

key1 = get_cache_key("CLT007994", "25078RA3EA", 7)
key2 = get_cache_key("CLT007994", "25078RA3EA", 7)

print("=== Reproduction du bug de clé de cache ===")
print(f"Appel 1 - quantite_suggeree=7   -> cache_key = {key1}")
print(f"Appel 2 - quantite_suggeree=300 -> cache_key = {key2}")
print(f"Les deux clés sont-elles identiques ? {key1 == key2}")
print()
print("-> Le cache ne contient NI la quantité, NI le score, NI le trend dans sa clé.")
print("-> Si l'explication de l'Appel 1 (7 unités) est mise en cache,")
print("   l'Appel 2 (300 unités, calculé plus tard) recevra le MEME TEXTE")
print("   d'explication pendant 24h, alors que le NOMBRE affiché à côté est différent.")


=== Reproduction du bug de clé de cache ===
Appel 1 - quantite_suggeree=7   -> cache_key = 9be71049278c313e2caee8622861db97
Appel 2 - quantite_suggeree=300 -> cache_key = 9be71049278c313e2caee8622861db97
Les deux clés sont-elles identiques ? True
-> Le cache ne contient NI la quantité, NI le score, NI le trend dans sa clé.
-> Si l'explication de l'Appel 1 (7 unités) est mise en cache,
   l'Appel 2 (300 unités, calculé plus tard) recevra le MEME TEXTE
   d'explication pendant 24h, alors que le NOMBRE affiché à côté est différent.


**C'est un vrai bug reproductible, pas une supposition.** La clé de cache actuelle est construite uniquement à partir de `client_id + code_article + month` — jamais de la quantité, du score de confiance, ou du trend. Concrètement :

```
09h00 — Premier appel : régresseur prédit 7 unités
        -> LLM génère "Baisse légère... 7 unités (Confiance 99%)"
        -> Stocké en cache sous la clé hash(CLT007994_25078RA3EA_7)

14h00 — Pipeline relancé (nouvelles données, ou nouvel entraînement)
        -> régresseur prédit maintenant 300 unités (le fameux cas Nokia)
        -> même client, même produit, même mois -> MÊME clé de cache
        -> le système sert le texte de 09h00, JAMAIS RÉGÉNÉRÉ
        -> résultat affiché : "300 unités" + texte parlant de "7 unités" et "baisse légère"
```

C'est **exactement** le genre d'incohérence qui rend une grosse prédiction encore plus suspecte et non-crédible aux yeux du commercial — le texte et le chiffre ne racontent plus la même histoire.


### Cache n°3 — Cache éventuel côté provider (Groq, etc.)

Certains providers d'inférence appliquent eux-mêmes un cache de requêtes strictement identiques (mêmes tokens en entrée, mêmes paramètres). C'est optionnel, dépend du provider, et **tu n'as aucun contrôle dessus ni aucune garantie qu'il existe**. Il n'explique pas le comportement observé dans ce projet — le Cache n°2 (le tien) est la seule explication qui colle exactement aux faits observés.


---
## 4. Comment forcer le système à produire une explication réellement pertinente

Deux problèmes distincts à corriger : **(a)** le cache peut servir un texte périmé, **(b)** même un texte frais du LLM ne référence jamais explicitement pourquoi un chiffre est gros ou petit.


### Fix (a) — Réparer la clé de cache pour inclure le contexte numérique

```python
def _get_cache_key(
    client_id: str,
    code_article: str,
    month: int,
    quantite_suggeree: int,
    score_confiance: float,
) -> str:
    # Le score est arrondi à 2 décimales pour absorber le bruit flottant
    # sans pour autant ignorer un vrai changement de prédiction.
    raw_str = f"{client_id}_{code_article}_{month}_{quantite_suggeree}_{round(score_confiance, 2)}"
    return hashlib.md5(raw_str.encode("utf-8")).hexdigest()
```

Avec ce changement, une nouvelle quantité prédite (300 au lieu de 7) génère une clé DIFFERENTE — donc le cache ne sert plus jamais un texte qui ne correspond pas au chiffre réellement affiché. C'est la correction la plus urgente, un changement d'une seule ligne avec un effet immédiat sur la fiabilité perçue du système.


### Fix (b) — Forcer le LLM (et le fallback) à expliquer explicitement les cas atypiques

Actuellement, ni le prompt LLM ni `_rule_based_explanation` ne comparent `quantite_suggeree` à l'historique réel du client. On peut calculer ce signal AVANT d'appeler le LLM, et le lui donner explicitement comme instruction.

```python
# Dans recommendation.py, avant d'appeler explain_suggestion :
max_qty_historique = float(row.get("max_qty", 1.0))
est_quantite_inhabituelle = sugg_qty > max_qty_historique * 1.5

explication = explain_suggestion(
    ...,
    quantite_suggeree=sugg_qty,
    max_qty_historique=max_qty_historique,       # nouveau paramètre
    est_quantite_inhabituelle=est_quantite_inhabituelle,  # nouveau paramètre
)
```

```python
# Dans explanation.py, enrichir le prompt :
alerte = (
    f"ATTENTION : cette quantité ({quantite_suggeree}) dépasse largement le maximum "
    f"historique de ce client pour ce produit ({max_qty_historique:.0f} unités). "
    f"Explique cette quantité en la reliant explicitement à cet historique, "
    f"ou signale-la comme une anomalie à vérifier par le commercial.\n"
) if est_quantite_inhabituelle else ""

prompt = (
    "Tu es un assistant IA pour une équipe de vente...\n"
    f"{alerte}"
    f"Quantité suggérée: {quantite_suggeree} unités\n"
    f"Quantité maximale déjà commandée par ce client pour ce produit: {max_qty_historique:.0f} unités\n"
    ...
)
```

Cette modification force explicitement le LLM à **soit justifier le chiffre par rapport à l'historique réel, soit signaler lui-même que c'est une anomalie à vérifier** — au lieu de générer une phrase générique de confiance qui masque le problème plutôt que de l'exposer.


---
## 5. Résumé — les 2 questions initiales, réponses directes

**"Qu'est-ce que le cache d'un LLM HuggingFace ?"**
→ Il n'y a pas UN cache — il y en a potentiellement 3 niveaux différents. Celui qui cause le comportement observé dans ce projet n'est **pas** un mécanisme du LLM ni de HuggingFace : c'est le dictionnaire Python `_explanation_cache` que le projet a lui-même codé dans `explanation.py`, avec une clé de cache incomplète qui ignore les valeurs numériques prédites.

**"Comment forcer une bonne description ?"**
→ Deux leviers complémentaires : corriger la clé de cache pour qu'elle inclue la quantité et le score (empêche de servir un texte périmé), puis enrichir le prompt avec une comparaison explicite à l'historique réel du client (force le LLM à justifier plutôt qu'à généraliser).
